In [1]:
!pip install iterative-stratification tqdm scikit-learn numpy torch transformers pandas scikit-multilear datetime

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement scikit-multilear (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for scikit-multilear


In [1]:
import datetime
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
from torch.optim import AdamW
import torch
from transformers import (
    AutoTokenizer,
    get_scheduler,
    DataCollatorWithPadding
)
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import csv

start_time = datetime.datetime.now()
print(f"Script started at {start_time}")

two_emotions = True
# Configuration metrics
file_name = "../GroundTruth.csv"
checkpoint = "bert-base-cased"
num_folds = 10
emotions = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral"]

if two_emotions:
    file_name = "../GroundTruthJoySadness.csv"
    emotions = ["Joy", "Sadness"]

# Data imbalance handling
undersample = False
cut_off = 150
class_weight = True
smote = False

# Build output filename dynamically based on options
suffixes = []
if undersample:
    suffixes.append("undersample")
if class_weight:
    suffixes.append("class_weight")
if smote:
    suffixes.append("smote")

suffix_str = "_" + "_".join(suffixes) if suffixes else ""
output_path_csv = f"results/results_{checkpoint}_with_folds_{num_folds}__{suffix_str}.csv"
if two_emotions:
    output_path_csv = f"results/results_2e_{checkpoint}_with_folds_{num_folds}__{suffix_str}.csv"


timing_info = {}
timing_info['Start timing'] = start_time

# ===================
# ===== DATASET =====
# ===================

# Read the original CSV file
df = pd.read_csv(file_name, quotechar='"', sep=';')
#df = df[df['Reject'] != 1]  # Filter out rows where Reject is 1

if undersample:
    start_undersampling = datetime.datetime.now()
    rng = np.random.default_rng(42)
    df = df.reset_index(drop=True)
    pos_counts = df[emotions].sum(axis=0).astype(int)
    targets = np.minimum(cut_off, pos_counts).astype(int)

    selected_idx = set()
    for lbl, tgt in zip(emotions, targets):
        if tgt <= 0:
            continue
        pos_idx = df.index[df[lbl] == 1].to_numpy()
        if len(pos_idx) <= tgt:
            chosen = pos_idx
        else:
            chosen = rng.choice(pos_idx, size=int(tgt), replace=False)
        selected_idx.update(int(x) for x in chosen)

    undersampled = df.loc[sorted(selected_idx)].reset_index(drop=True)
    print("Target per label:", dict(zip(emotions, targets.tolist())))
    print("Result per-label counts:\n", undersampled[emotions].sum().to_dict())
    print("Total rows selected:", len(undersampled))
    df = undersampled
    end_undersampling = datetime.datetime.now()
    timing_info['Undersampling time'] = end_undersampling - start_undersampling

X = df[['sentence']].values
y = df[emotions].values

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pos_weight = None

if class_weight:
    start_time_class_weight = datetime.datetime.now()
    label_counts = df[emotions].sum().values
    median_count = np.median(label_counts)

    label_weights = median_count / (len(emotions) * label_counts + 1e-6)
    pos_weight = torch.tensor(label_weights, dtype=torch.float).to(device)

    print("Label counts:\n", df[emotions].sum())
    print("Computed weights:\n", pos_weight)

    print("🔹 Using smoothed class weights:")
    for emotion, w in zip(emotions, pos_weight.tolist()):
        print(f"{emotion:12s}: {w:.3f}")

    timing_info['Class weight calculation time'] = datetime.datetime.now() - start_time_class_weight


# --------- custom Dataset (returns raw lists so collator can pad) ---------- #
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        labels = [int(row[l]) for l in self.label_names]
        tokenized["labels"] = labels
        return tokenized


def collate_fn(batch):
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.float)
    return batch_inputs


def create_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(emotions),
        problem_type="multi_label_classification"
    )
    model.to(device)
    return model


# ============================
# ===== CROSS VALIDATION =====
# ============================

multilabel_stratified_in_folds = MultilabelStratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
all_metrics = []
fold_results = []

for fold, (train_idx, val_idx) in enumerate(multilabel_stratified_in_folds.split(X, y)):
    timing_fold = datetime.datetime.now()
    print(f"\n===== Fold {fold + 1} =====")
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = EmotionDataset(train_df, tokenizer, emotions)
    val_dataset = EmotionDataset(val_df, tokenizer, emotions)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

    model = create_model()

    optimizer = AdamW(model.parameters(), lr=2e-5)
    num_epochs = 3
    num_training_steps = num_epochs * len(train_loader)
    lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0,
                                 num_training_steps=num_training_steps)

    # ---- training loop ----
    model.train()
    progress_bar = tqdm(range(num_training_steps))

    if class_weight and pos_weight is not None:
        loss_fct = torch.nn.CrossEntropyLoss(weight=pos_weight)
    else:
        loss_fct = torch.nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            if class_weight:
                logits = outputs.logits
                loss = loss_fct(logits, batch["labels"])
            else:
                loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

    # ---- evaluation ----
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch["labels"].cpu().numpy().astype(int)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            outputs = model(**inputs)
            logits = outputs.logits.cpu()
            preds = (torch.sigmoid(logits).numpy() > 0.5).astype(int)
            all_preds.append(preds)
            all_labels.append(labels)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    sentences = val_df["sentence"].values


    def binvec_to_names(vec):
        names = [emotions[i] for i, v in enumerate(vec) if int(v) == 1]
        return ", ".join(names) if names else "None"


    true_emotions = [binvec_to_names(row) for row in y_true]
    pred_emotions = [binvec_to_names(row) for row in y_pred]

    df_predictions = pd.DataFrame({
        "sentence": sentences,
        "true_emotions": true_emotions,
        "pred_emotions": pred_emotions
    })

    # show a quick preview and optionally save per fold
    print(df_predictions.head(20).to_string(index=False))
    df_predictions.to_csv(f"predictions/predictions_fold_{fold + 1}_out_of_{num_folds}.csv", index=False)

    fold_results.append({
        "fold": fold + 1,
        "y_true": y_true,
        "y_pred": y_pred
    })

    metrics = {
        "Subset Accuracy (Exact Match)": accuracy_score(y_true, y_pred),
        "Micro Precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro Recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro F1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }

    # --- Print the Summary ---
    print(f"\n=== Metrics for fold {fold + 1} ===")
    for k, v in metrics.items():
        print(f"{k:25s}: {v:.4f}")

    all_metrics.append(metrics)
    end_timing_fold = datetime.datetime.now()
    timing_info[f'Fold {fold + 1} time'] = end_timing_fold - timing_fold

print("\n===== Summary per fold =====")
timing_results = datetime.datetime.now()

# Create the overall metrics table
df_metrics = pd.DataFrame(all_metrics).T  # transpose to have metrics as rows
df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
print("\n=== Overall Metrics per Fold ===")
print(df_metrics.to_markdown(floatfmt=".4f"))

# ===== PER-EMOTION METRICS =====
emotion_metrics = {emotion: {"Precision": [], "Recall": [], "F1": [], "Accuracy": []} for emotion in emotions}

for fold_data in fold_results:
    y_true = fold_data["y_true"]
    y_pred = fold_data["y_pred"]
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    acc = [accuracy_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]

    for i, emotion in enumerate(emotions):
        emotion_metrics[emotion]["Precision"].append(prec[i])
        emotion_metrics[emotion]["Recall"].append(rec[i])
        emotion_metrics[emotion]["F1"].append(f1[i])
        emotion_metrics[emotion]["Accuracy"].append(acc[i])

print("\n===== Per Emotion Metrics per Fold =====")
for emotion, metrics_dict in emotion_metrics.items():
    df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
    print(f"\n--- {emotion} ---")
    print(df_emotion.to_markdown(floatfmt=".4f"))

# ===== Average results per emotion =====
print("\n===== Average Per-Emotion Metrics =====")
df_emotion_avg = pd.DataFrame({
    emotion: {metric: np.mean(values) for metric, values in metrics.items()}
    for emotion, metrics in emotion_metrics.items()
})
print(df_emotion_avg.T.to_markdown(floatfmt=".4f"))

# ===== Average results =====
df_metrics = pd.DataFrame(all_metrics)
print("\n===== Cross-validation mean results =====")
print(df_metrics.mean())

timing_info[f'Printing results'] = datetime.datetime.now() - timing_results

with open(output_path_csv, mode="w", newline="") as f:
    writer = csv.writer(f)

    # ===== SUMMARY PER FOLD =====
    writer.writerow(["===== Summary per fold ====="])
    df_metrics = pd.DataFrame(all_metrics).T  # metrics as rows
    df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
    df_metrics.reset_index(inplace=True)
    df_metrics.rename(columns={"index": "Metric"}, inplace=True)
    df_metrics.to_csv(f, index=False, float_format="%.4f")
    writer.writerow([])  # blank line

    # ===== PER-EMOTION METRICS PER FOLD =====
    writer.writerow(["===== Per Emotion Metrics per Fold ====="])
    for emotion, metrics_dict in emotion_metrics.items():
        writer.writerow([f"--- {emotion} ---"])
        df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
        df_emotion.reset_index(inplace=True)
        df_emotion.rename(columns={"index": "Metric"}, inplace=True)
        df_emotion.to_csv(f, index=False, float_format="%.4f")
        writer.writerow([])  # blank line

    # ===== AVERAGE PER EMOTION =====
    writer.writerow(["===== Average Per-Emotion Metrics ====="])
    df_emotion_avg = pd.DataFrame({
        emotion: {metric: np.mean(values) for metric, values in metrics.items()}
        for emotion, metrics in emotion_metrics.items()
    })
    df_emotion_avg = df_emotion_avg.T.reset_index().rename(columns={"index": "Emotion"})
    df_emotion_avg.to_csv(f, index=False, float_format="%.4f")
    writer.writerow([])

    # ===== CROSS-VALIDATION MEAN RESULTS =====
    writer.writerow(["===== Cross-validation Mean Results ====="])
    df_metrics = pd.DataFrame(all_metrics)
    df_mean = df_metrics.mean().to_frame(name="Mean").reset_index().rename(columns={"index": "Metric"})
    df_mean.to_csv(f, index=False, float_format="%.4f")

    # ===== CLASS WEIGHTS =====
    if class_weight:
        writer.writerow([])
        writer.writerow(["===== Class Weights per Emotion ====="])
        writer.writerow(["Emotion", "Weight"])
        for emotion, weight in zip(emotions, pos_weight.tolist()):
            writer.writerow([emotion, f"{weight:.6f}"])
        writer.writerow([])

    writer.writerow(["===== Timings ====="])
    for k, v in timing_info.items():
        writer.writerow([k, str(v)])

    writer.writerow(["Total duration", str(datetime.datetime.now() - start_time)])

print(f"\n✅ All tables saved in one CSV file: {os.path.abspath(output_path_csv)}")


ModuleNotFoundError: No module named 'iterstrat'

In [4]:
import datetime
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

start_time = datetime.datetime.now()
print(f"Script started at {start_time}")

# Configuration metrics
file_name = "../GroundTruth.csv"
checkpoint = "bert-base-cased"
num_folds = 5
emotions = ['Joy', 'Trust', 'Fear', 'Surprise', 'Sadness', 'Disgust', 'Anger', 'Anticipation', 'Neutral']
label_columns = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral", "Reject"]

undersample = True
cut_off = 150
suffixes = []
if undersample:
    suffixes.append("undersample")

suffix_str = "_" + "_".join(suffixes) if suffixes else ""
output_path_csv = f"../resultsBDS/binary_results_{checkpoint}_with_folds_{num_folds}__{suffix_str}.csv"

df_general = pd.read_csv(file_name, sep=";")
df_general = df_general[df_general['Reject'] != 1]

if undersample:
    start_undersampling = datetime.datetime.now()
    rng = np.random.default_rng(42)
    df_general = df_general.reset_index(drop=True)
    pos_counts = df_general[emotions].sum(axis=0).astype(int)
    targets = np.minimum(cut_off, pos_counts).astype(int)

    selected_idx = set()
    for lbl, tgt in zip(emotions, targets):
        if tgt <= 0:
            continue
        pos_idx = df_general.index[df_general[lbl] == 1].to_numpy()
        if len(pos_idx) <= tgt:
            chosen = pos_idx
        else:
            chosen = rng.choice(pos_idx, size=int(tgt), replace=False)
        selected_idx.update(int(x) for x in chosen)

    undersampled = df_general.loc[sorted(selected_idx)].reset_index(drop=True)
    print("Target per label:", dict(zip(emotions, targets.tolist())))
    print("Result per-label counts:\n", undersampled[emotions].sum().to_dict())
    print("Total rows selected:", len(undersampled))
    df_general = undersampled
    end_undersampling = datetime.datetime.now()
    timing_info['Undersampling time'] = end_undersampling - start_undersampling

non_label_columns = [col for col in df_general.columns if col not in emotions]
all_results = []
timing_info = {}

# Dataset for single-label
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_col):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        tokenized["labels"] = int(row[self.label_col])
        return tokenized

def collate_fn(batch):
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch_inputs

for label in emotions:
    start_time_label = datetime.datetime.now()
    label_col = 'Emotion'
    df = df_general[non_label_columns + [label]].copy()
    df = df.rename(columns={label: label_col})
    output_filename = f"../binaryDS/{label}.csv"
    df.to_csv(output_filename, sep=";", index=False)

    print(f"Archivo generado: {output_filename}")
    timing_info[f'Generate {label} file'] = datetime.datetime.now() - start_time_label


    print("----------------------------------------------------")
    print(f"Processing label: {label}")
    print("----------------------------------------------------")

    num_labels = df[label_col].nunique()

    # Tokenizer and collator
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cross-validation
    X = df["sentence"].values
    y = df[label_col].values
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    all_metrics = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        start_time_fold = datetime.datetime.now()
        print(f"\n===== Fold {fold + 1} =====")
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)

        train_dataset = EmotionDataset(train_df, tokenizer, label_col)
        val_dataset = EmotionDataset(val_df, tokenizer, label_col)

        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint,
            num_labels=num_labels
        )
        model.to(device)

        optimizer = AdamW(model.parameters(), lr=2e-5)
        num_epochs = 3
        num_training_steps = num_epochs * len(train_loader)
        lr_scheduler = get_scheduler("linear", optimizer=optimizer,
                                     num_warmup_steps=0,
                                     num_training_steps=num_training_steps)

        model.train()
        progress_bar = tqdm(range(num_training_steps))
        for epoch in range(num_epochs):
            for batch in train_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
                progress_bar.update(1)

        # Evaluation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                labels = batch["labels"].cpu().numpy()
                inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
                outputs = model(**inputs)
                preds = outputs.logits.argmax(dim=-1).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(labels)

        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_labels)

        global_metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "micro-precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
            "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "micro-recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
            "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "micro-f1": f1_score(y_true, y_pred, average="micro", zero_division=0)
        }
        print(f"Accuracy  -> {global_metrics['accuracy']}")
        print(f"Precision -> Macro: {global_metrics['precision']}, Micro: {global_metrics['micro-precision']}")
        print(f"Recall    -> Macro: {global_metrics['recall']}, Micro: {global_metrics['micro-recall']}")
        print(f"F1 Score  -> Macro: {global_metrics['f1']}, Micro: {global_metrics['micro-f1']}")

        result_row = {"emotion": label, "fold": fold+1}
        result_row.update(global_metrics)
        all_results.append(result_row)

        report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
        df_report = pd.DataFrame(report).transpose()
        df_report["fold"] = fold + 1
        all_metrics.append((global_metrics, df_report))
        timing_info[f'{label} - Fold {fold+1}'] = datetime.datetime.now() - start_time_fold


    # Average results
    global_results = [m[0] for m in all_metrics]
    df_global = pd.DataFrame(global_results)
    print(f"\n===== Global metrics averaged across folds {label}=====")
    print(df_global.mean())


# === Simplified results summary ===
df_summary = pd.DataFrame(all_results)

print("\n===== Results by Emotion =====")
emotion_tables = []

for emotion in df_summary["emotion"].unique():
    sub = df_summary[df_summary["emotion"] == emotion]
    # Build a simple table: metrics as rows, folds as columns
    metrics = ["accuracy", "precision", "recall", "f1"]
    table = pd.DataFrame(index=metrics)
    for fold in sorted(sub["fold"].unique()):
        fold_values = sub[sub["fold"] == fold].iloc[0]
        table[f"Fold {fold}"] = [fold_values[m] for m in metrics]

    print(f"\nEmotion: {emotion}")
    print(table)
    emotion_tables.append((emotion, table))

# === Timings summary ===
print("\n===== Timings =====")
timing_rows = []
for key, delta in timing_info.items():
    if "Fold" in key:
        emotion = key.split(" - ")[0]
        fold = key.split("Fold ")[1]
        timing_rows.append({
            "Emotion": emotion,
            "Fold": f"Fold {fold}",
            "Time (s)": delta.total_seconds()
        })

df_timing = pd.DataFrame(timing_rows)

for emotion in df_timing["Emotion"].unique():
    sub = df_timing[df_timing["Emotion"] == emotion].pivot(index="Emotion", columns="Fold", values="Time (s)")
    print(f"\nEmotion: {emotion}")
    print(sub)

# === Save all to CSV ===
with open(output_path_csv, "w", encoding="utf-8") as f:
    for emotion, table in emotion_tables:
        f.write(f"\nEmotion: {emotion}\n")
        table.to_csv(f, float_format="%.4f")
    f.write("\n\n===== Timings =====\n")
    df_timing.to_csv(f, float_format="%.2f", index=False)

print(f"\n✅ Results saved to {output_path_csv}")

Script started at 2025-11-26 11:05:23.074085
Target per label: {'Joy': 150, 'Trust': 150, 'Fear': 24, 'Surprise': 54, 'Sadness': 150, 'Disgust': 50, 'Anger': 27, 'Anticipation': 150, 'Neutral': 88}
Result per-label counts:
 {'Joy': 191, 'Trust': 150, 'Fear': 24, 'Surprise': 54, 'Sadness': 179, 'Disgust': 50, 'Anger': 27, 'Anticipation': 158, 'Neutral': 88}
Total rows selected: 776


NameError: name 'timing_info' is not defined